# Laboratorio CNNs-XAI

Cuadernillo organizado para los siete ejercicios del laboratorio. Las respuestas analíticas aparecen en celdas `markdown` para que puedan copiarse luego en la parte manuscrita.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RESULTS_PATH = ROOT / 'reports' / 'metrics' / 'lab_results.json'
results = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
results

## Ejercicio 1. Descarga y exploración del dataset

### Respuesta para la parte manuscrita

El dataset contiene **2720 imágenes de la clase male** y **2698 imágenes de la clase female**, para un total de **5418 imágenes**. La colección presenta una variabilidad alta en tamaño, formato y modo de color. En `male`, los anchos van de **194** a **5472** píxeles, mientras que en `female` van de **236** a **8675**. También se encontraron imágenes en modos `RGB`, `RGBA`, `L` y `P`, lo que justificó convertir todo explícitamente a `RGB` antes del entrenamiento.

### Estructura de carpetas solicitada

```text
data/
├── male/
└── female/
```

### Observaciones clave

- Hay mezcla de extensiones `.jpg`, `.jpeg` y `.png`.
- Existen tamaños muy dispares, por lo que era obligatorio unificar la entrada.
- No se detectaron archivos corruptos en la inspección inicial.

In [ ]:
display(Image(filename=str(ROOT / 'reports' / 'figures' / 'class_distribution.png')))
display(Image(filename=str(ROOT / 'reports' / 'figures' / 'dataset_mosaic.png')))

## Ejercicio 2. Preprocesamiento y partición

### Respuesta para la parte manuscrita

El preprocesamiento consistió en: conversión a `RGB`, ajuste de tamaño uniforme con preservación del contenido facial mediante `resize_with_pad`, normalización al rango `[0, 1]` y partición estratificada en entrenamiento, validación y prueba. Para la fase de modelado se tomó una **submuestra estratificada de 1200 imágenes** (**600 por clase**) para mantener un entrenamiento ligero y desplegable en Streamlit Cloud. La partición final quedó en **840 imágenes para entrenamiento**, **180 para validación** y **180 para prueba**.

### Esquema del flujo

```text
lectura de rutas -> conversión a RGB -> resize con padding -> normalización -> partición estratificada
```

### Justificación conceptual

- Mantener color es importante porque la red aprende patrones cromáticos y de textura.
- Mantener tamaño uniforme es importante porque las capas convolucionales y densas requieren tensores consistentes.
- La partición estratificada evita desbalance entre clases en los subconjuntos.

In [ ]:
pd.DataFrame([
    {
        'train': results['split_counts']['train'],
        'validation': results['split_counts']['validation'],
        'test': results['split_counts']['test'],
        'modeling_dataset_count': results['modeling_dataset_count'],
    }
])

## Ejercicio 3. Construcción y entrenamiento de la CNN

### Respuesta para la parte manuscrita

La arquitectura final corresponde a una CNN secuencial construida desde cero con bloques `Conv2D + BatchNormalization + MaxPooling2D`, seguida por una capa `GlobalAveragePooling2D`, una capa densa oculta y una salida sigmoide para clasificación binaria.

### Descripción resumida de la arquitectura

- Entrada: imagen RGB de `96x96`.
- Bloques convolucionales principales: filtros `[8, 16, 32]`.
- Dropout seleccionado: `0.35`.
- Salida: `Dense(1, activation='sigmoid')`.

### Análisis de entrenamiento

En prueba, el modelo final obtuvo una **accuracy de 59.44%** y una **loss de 0.6878**. Las curvas de entrenamiento y validación permiten verificar si la red converge de manera estable y si aparece separación excesiva entre ambas curvas, lo que sería señal de sobreajuste. Las curvas muestran un comportamiento más cercano al subajuste que al sobreajuste, porque la accuracy se mantiene moderada y la pérdida sigue relativamente alta tanto en entrenamiento como en validación.

In [ ]:
print((ROOT / 'reports' / 'metrics' / 'model_summary.txt').read_text(encoding='utf-8'))
display(Image(filename=str(ROOT / 'reports' / 'figures' / 'training_final.png')))

## Ejercicio 4. Ajuste de hiperparámetros

### Respuesta para la parte manuscrita

Se compararon al menos dos configuraciones distintas de hiperparámetros. La decisión final se tomó a partir de la métrica de validación, priorizando la mejor `accuracy` y, en caso de cercanía, la menor `loss`.

### Tabla comparativa

| Configuración | Filtros | Dropout | Learning rate | Mejor época | Mejor val accuracy | Mejor val loss |
|---|---:|---:|---:|---:|---:|---:|
| regularized | [8, 16, 32] | 0.35 | 0.0005 | 2 | 0.5778 | 0.6924 |
| baseline | [8, 16, 32] | 0.2 | 0.001 | 1 | 0.5722 | 0.6928 |

La mejor configuración fue **regularized**, con una mejor `val_accuracy` de **0.5778** y una `val_loss` de **0.6924**. Se seleccionó porque mostró el mejor equilibrio entre rendimiento y estabilidad durante la validación. Aun así, el margen entre configuraciones fue pequeño, lo que sugiere que el principal límite estuvo más asociado a la capacidad total del modelo ligero y al tamaño de la submuestra que a una sola decisión de hiperparámetros.

In [ ]:
display(Image(filename=str(ROOT / 'reports' / 'figures' / 'hyperparameter_comparison.png')))
pd.read_csv(ROOT / 'reports' / 'metrics' / 'hyperparameter_results.csv')

## Ejercicio 5. Interpretabilidad visual

### Respuesta para la parte manuscrita

El **Saliency Map** resalta los píxeles individuales cuya perturbación cambia más la salida del modelo. Por eso suele verse más granular y sensible al detalle fino. Por su parte, **Grad-CAM** utiliza las activaciones de una capa convolucional profunda para identificar regiones espaciales completas, por lo que tiende a producir mapas más estructurados y fáciles de interpretar a nivel de zonas del rostro.

Para el ejemplo seleccionado, la imagen pertenecía a la clase real **male** y la predicción fue **male**, con probabilidad de `male` igual a **0.5402**. La interpretación debe centrarse en si ambos mapas privilegian ojos, cejas, contorno facial, nariz o cabello, en lugar de activar el fondo o los bordes externos.

In [ ]:
display(Image(filename=str(ROOT / 'reports' / 'figures' / 'xai_example.png')))
results['xai_sample']

## Ejercicio 6. Despliegue con Streamlit

### Respuesta para la parte manuscrita

La interfaz de Streamlit se diseñó con tres bloques principales:

1. Área de carga de imagen.
2. Panel de predicción con probabilidades por clase.
3. Área de visualización de interpretabilidad con `Saliency Map` y `Grad-CAM`.

### Flujo entre interfaz y modelo

```text
imagen cargada -> preprocesamiento -> predicción sigmoide -> cálculo de Saliency Map y Grad-CAM -> visualización
```

### Dos mejoras posibles

- Incorporar una galería de ejemplos de prueba para demostraciones rápidas en clase.
- Añadir un módulo de comparación entre varias imágenes o una sección de limitaciones/sesgos del modelo.

In [ ]:
from pathlib import Path
app_path = ROOT / 'app' / 'streamlit_app.py'
print(app_path)
print(app_path.read_text(encoding='utf-8')[:2000])

## Ejercicio 7. Presentación y reflexión final

### Guion breve para presentar en clase

1. Mostrar la app funcionando con una imagen de rostro autorizada.
2. Explicar la probabilidad generada por la salida sigmoide.
3. Comparar el mapa de Saliency con Grad-CAM.
4. Señalar si el modelo se concentró en rasgos faciales plausibles.

### Reflexión final

Este laboratorio muestra que una CNN construida desde cero puede resolver una tarea binaria de clasificación facial, pero también deja claro que la interpretación del modelo es indispensable para verificar si la decisión se apoya en rasgos razonables. La combinación de predicción + XAI + despliegue convierte el trabajo en una solución más completa y defendible en presentación.

In [ ]:
# Si desea regenerar todos los resultados desde cero, ejecute estas líneas manualmente.
# import os, subprocess
# subprocess.run(['python', 'scripts/train_pipeline.py'], cwd=str(ROOT), check=True)
# subprocess.run(['python', 'scripts/generate_notebook.py'], cwd=str(ROOT), check=True)
